# LLaVA — Zero-shot evaluation

## 1. Imports

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm
import torch

from dual_resnet_shared import build_synth_index
from llava_shared import (
    load_llava_model, parse_emotion,
    evaluate_llava_results, run_kfold, plot_kfold_results,
    MAX_NEW_TOKENS,
)

warnings.filterwarnings("ignore")
print(f"CUDA avaiable: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {torch.cuda.get_device_name(0)} — {vram:.1f} GB VRAM")

## 2. Config

In [ ]:
# EMOTIC
PARQUET_EMOTIC = Path("emotic_basic.parquet")
EMOTIC_BASE = Path("datasets/emotic")
OUTPUT_EMOTIC = Path("emotic_llava_zeroshot_test.parquet")
EMOTIC_EMOTIONS = ["Happiness", "Sadness", "Anger", "Fear", "Surprise", "Aversion"]
KFOLD_PARQUET_EMOTIC = Path("emotic_llava_zeroshot.parquet")

# NCAERS
NCAERS_ROOT = Path("datasets/NCAERS")
INDEX_NCAERS = NCAERS_ROOT / "ncaers_index.parquet"
OUTPUT_NCAERS = Path("ncaers_llava_zeroshot_test.parquet")
NCAERS_EMOTIONS = ["Anger", "Disgust", "Fear", "Happy", "Neutral", "Sad", "Surprise"]
KFOLD_PARQUET_NCAERS = Path("ncaers_llava_zeroshot.parquet")

# SynthCAER
SYNTH_ROOT = Path("SynthCAER")
OUTPUT_SYNTH = Path("synth_llava_zeroshot_test.parquet")
SYNTH_EMOTIONS = ["Angry", "Disgust", "Fear", "Happy", "Neutral", "Sad", "Surprise"]
SYNTH_ALIASES = {
    "anger": "Angry", "angry": "Angry",
    "disgust": "Disgust", "disgusted": "Disgust", "aversion": "Disgust",
    "fear": "Fear", "fearful": "Fear", "scared": "Fear",
    "happy": "Happy", "happiness": "Happy", "joy": "Happy",
    "neutral": "Neutral",
    "sad": "Sad", "sadness": "Sad", "sorrow": "Sad",
    "surprise": "Surprise", "surprised": "Surprise",
}

SAMPLE_N    = None
RANDOM_SEED = 42
N_FOLDS     = 5

## 3. Load LLaVA

In [ ]:
@torch.inference_mode()
# LLaVA pipeline
def _run_llava(image, prompt):
    inputs = processor(text=prompt, images=image, return_tensors="pt").to(device)
    out    = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
    return processor.decode(out[0], skip_special_tokens=True)

# Inference loop
def run_inference(df, get_image, build_row, prompt, emotions, aliases=None, output_path=None, tag=""):
    results, errors = [], []
    for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Running {tag}"):
        try:
            image = get_image(row)
            if image is None:
                errors.append({"error": "file_not_found"})
                continue
            raw       = _run_llava(image, prompt)
            predicted = parse_emotion(raw, emotions, aliases)
            results.append(build_row(row, predicted, raw))
        except Exception as e:
            errors.append({"error": str(e)})

    if not results:
        print(f"ERROR: no results — {len(errors):,} errors.")

    df_res = pd.DataFrame(results)
    if output_path:
        df_res.to_parquet(output_path, index=False)
        print(f"Saved in: {output_path}")
    n_unp = df_res["unparseable"].sum()
    print(f"Processed: {len(df_res):,}  |  Errors: {len(errors):,}  |  Without parse: {n_unp:,}")
    return df_res

In [ ]:
model, processor, device = load_llava_model()

## 4. EMOTIC

### Load dataset

In [ ]:
df_emotic = pd.read_parquet(PARQUET_EMOTIC)
df_emotic = df_emotic[df_emotic["split"] == "test"].reset_index(drop=True)
if SAMPLE_N:
    df_emotic = df_emotic.sample(n=min(SAMPLE_N, len(df_emotic)),
                                  random_state=RANDOM_SEED).reset_index(drop=True)
print(f"Cases test EMOTIC: {len(df_emotic):,}")
print(df_emotic["basic_emotion"].value_counts().to_string())

### Run inference

In [ ]:
_emotic_emo_str = ", ".join(EMOTIC_EMOTIONS)
EMOTIC_PROMPT = (
    f"USER: <image>\n"
    f"Look at the person in this image and identify their primary emotion. "
    f"Choose exactly one from the following options: {_emotic_emo_str}. "
    f"Reply with only the emotion name, nothing else.\n"
    f"ASSISTANT:"
)

def emotic_load(row):
    p = EMOTIC_BASE / row["folder"] / row["filename"]
    return Image.open(p).convert("RGB") if p.exists() else None

def emotic_row(row, pred, raw):
    return {
        "split": row["split"], "folder": row["folder"], "filename": row["filename"],
        "true_emotion": row["basic_emotion"], "predicted_emotion": pred,
        "raw_response": raw,
        "correct": pred == row["basic_emotion"] if pred else False,
        "unparseable": pred is None,
    }

_REQUIRED = {"true_emotion", "predicted_emotion", "unparseable"}

# Load results or execute inference (first time)
if OUTPUT_EMOTIC.exists():
    _df = pd.read_parquet(OUTPUT_EMOTIC)
    if _REQUIRED.issubset(_df.columns):
        df_emotic_res = _df
        print(f"Loading {OUTPUT_EMOTIC}... {len(df_emotic_res):,} cases.")
    else:
        print(f"Incomplete data (missing: {_REQUIRED - set(_df.columns)}). Executing inference.")
        df_emotic_res = run_inference(
            df_emotic, emotic_load, emotic_row,
            EMOTIC_PROMPT, EMOTIC_EMOTIONS, output_path=OUTPUT_EMOTIC, tag="EMOTIC",
        )
else:
    df_emotic_res = run_inference(
        df_emotic, emotic_load, emotic_row,
        EMOTIC_PROMPT, EMOTIC_EMOTIONS, output_path=OUTPUT_EMOTIC, tag="EMOTIC",
    )

### Evaluation (test split)

In [ ]:
emotic_metrics = evaluate_llava_results(
    df_emotic_res, EMOTIC_EMOTIONS,
    tag="EMOTIC test split",
    save_stem="llava_v3_emotic",
)

### K-fold Robust analysis (full dataset)

In [ ]:
if KFOLD_PARQUET_EMOTIC.exists():
    _df = pd.read_parquet(KFOLD_PARQUET_EMOTIC)
    df_kf_emotic = _df
    print(f"K-fold: using {KFOLD_PARQUET_EMOTIC} ({len(df_kf_emotic):,} cases).")
else:
    (f"{KFOLD_PARQUET_EMOTIC} not found. Using test split.")
    df_kf_emotic = df_emotic_res.copy()

df_kf_emotic = df_kf_emotic[~df_kf_emotic["unparseable"]].reset_index(drop=True)
print(f"Cases per fold: {len(df_kf_emotic):,}")
print(df_kf_emotic["true_emotion"].value_counts().to_string())

df_folds_emotic = run_kfold(df_kf_emotic, EMOTIC_EMOTIONS, n_folds=N_FOLDS, random_seed=RANDOM_SEED)
print(df_folds_emotic.to_string(index=False, float_format="{:.3f}".format))

In [ ]:
metric_cols = ["Accuracy", "Macro F1"] + [f"F1_{e}" for e in EMOTIC_EMOTIONS]
df_stats_emotic = df_folds_emotic[metric_cols].agg(["mean", "std", "min", "max"])
df_stats_emotic.index = ["Media", "Std", "Min", "Max"]
print(df_stats_emotic.T.to_string(float_format="{:.3f}".format))

In [ ]:
plot_kfold_results(
    df_folds_emotic, EMOTIC_EMOTIONS,
    tag="EMOTIC",
    save_prefix="llava_v3_emotic",
)

## 5. NCAERS

### Load dataset

In [ ]:
df_ncaers = pd.read_parquet(INDEX_NCAERS)
df_ncaers = df_ncaers[df_ncaers["split"] == "test"].reset_index(drop=True)
if SAMPLE_N:
    df_ncaers = df_ncaers.sample(n=min(SAMPLE_N, len(df_ncaers)),
                                   random_state=RANDOM_SEED).reset_index(drop=True)
print(f"Cases test NCAERS: {len(df_ncaers):,}")
print(df_ncaers["label"].value_counts().to_string())

### Run inference

In [ ]:
_ncaers_emo_str = ", ".join(NCAERS_EMOTIONS)
NCAERS_PROMPT = (
    f"USER: <image>\n"
    f"Look at the person in this image and identify their primary emotion. "
    f"Choose exactly one from the following options: {_ncaers_emo_str}. "
    f"Reply with only the emotion name, nothing else.\n"
    f"ASSISTANT:"
)

def ncaers_load(row):
    p = NCAERS_ROOT / row["split"] / row["label"] / row["filename"]
    return Image.open(p).convert("RGB") if p.exists() else None

def ncaers_row(row, pred, raw):
    return {
        "split": row["split"], "label": row["label"],
        "video_stem": row.get("video_stem", ""), "filename": row["filename"],
        "true_emotion": row["label"], "predicted_emotion": pred,
        "raw_response": raw,
        "correct": pred == row["label"] if pred else False,
        "unparseable": pred is None,
    }

if OUTPUT_NCAERS.exists():
    _df = pd.read_parquet(OUTPUT_NCAERS)
    if _REQUIRED.issubset(_df.columns):
        df_ncaers_res = _df
        print(f"Loading {OUTPUT_NCAERS}... {len(df_ncaers_res):,} cases.")
    else:
        print(f"Incomplete data (missing: {_REQUIRED - set(_df.columns)}). Executing inference.")
        df_ncaers_res = run_inference(
            df_ncaers, ncaers_load, ncaers_row,
            NCAERS_PROMPT, NCAERS_EMOTIONS, output_path=OUTPUT_NCAERS, tag="NCAERS",
        )
else:
    df_ncaers_res = run_inference(
        df_ncaers, ncaers_load, ncaers_row,
        NCAERS_PROMPT, NCAERS_EMOTIONS, output_path=OUTPUT_NCAERS, tag="NCAERS",
    )

### Evaluation

In [ ]:
ncaers_metrics = evaluate_llava_results(
    df_ncaers_res, NCAERS_EMOTIONS,
    tag="NCAERS test split",
    save_stem="llava_ncaers_v2",
)

### K-fold Robust analysis (full dataset)

In [ ]:
if KFOLD_PARQUET_NCAERS.exists():
    _df = pd.read_parquet(KFOLD_PARQUET_NCAERS)
    df_kf_ncaers = _df
    print(f"K-fold: using {KFOLD_PARQUET_NCAERS} ({len(df_kf_ncaers):,} cases).")
else:
    print(f"{KFOLD_PARQUET_NCAERS} not found. Using test split.")
    df_kf_ncaers = df_ncaers_res.copy()

df_kf_ncaers = df_kf_ncaers[~df_kf_ncaers["unparseable"]].reset_index(drop=True)
print(f"Cases per fold: {len(df_kf_ncaers):,}")
print(df_kf_ncaers["true_emotion"].value_counts().to_string())

df_folds_ncaers = run_kfold(df_kf_ncaers, NCAERS_EMOTIONS, n_folds=N_FOLDS, random_seed=RANDOM_SEED)
print(df_folds_ncaers.to_string(index=False, float_format="{:.3f}".format))

In [ ]:
metric_cols = ["Accuracy", "Macro F1"] + [f"F1_{e}" for e in NCAERS_EMOTIONS]
df_stats_ncaers = df_folds_ncaers[metric_cols].agg(["mean", "std", "min", "max"])
df_stats_ncaers.index = ["Media", "Std", "Min", "Max"]
print(df_stats_ncaers.T.to_string(float_format="{:.3f}".format))

In [ ]:
plot_kfold_results(
    df_folds_ncaers, NCAERS_EMOTIONS,
    tag="NCAER-S",
    save_prefix="llava_ncaers_v2",
)

## 6. SynthCAER

### Load dataset

In [ ]:
df_synth = build_synth_index()
df_synth = df_synth[df_synth["split"] == "test"].reset_index(drop=True)
if SAMPLE_N:
    df_synth = df_synth.sample(n=min(SAMPLE_N, len(df_synth)),
                                random_state=RANDOM_SEED).reset_index(drop=True)
print(f"Cases test SynthContext: {len(df_synth):,}")
print(df_synth["emotion"].value_counts().to_string())

### Run inference

In [ ]:
_synth_emo_str = ", ".join(SYNTH_EMOTIONS)
SYNTH_PROMPT = (
    f"USER: <image>\n"
    f"Look at the person in this image and identify their primary emotion. "
    f"Choose exactly one from the following options: {_synth_emo_str}. "
    f"Reply with only the emotion name, nothing else.\n"
    f"ASSISTANT:"
)

def synth_load(row):
    p = SYNTH_ROOT / row["rel_path"]
    return Image.open(p).convert("RGB") if p.exists() else None

def synth_row(row, pred, raw):
    return {
        "seed": row["seed"], "context": row["context"],
        "emotion": row["emotion"], "filename": row["filename"],
        "rel_path": row["rel_path"], "split": row["split"],
        "true_emotion": row["emotion"], "predicted_emotion": pred,
        "raw_response": raw,
        "correct": pred == row["emotion"] if pred else False,
        "unparseable": pred is None,
    }

if OUTPUT_SYNTH.exists():
    _df = pd.read_parquet(OUTPUT_SYNTH)
    if _REQUIRED.issubset(_df.columns):
        df_synth_res = _df
        print(f"Loading {OUTPUT_SYNTH}... {len(df_synth_res):,} cases.")
    else:
        print(f"Incomplete dataset (missing: {_REQUIRED - set(_df.columns)}). Executing inference.")
        df_synth_res = run_inference(
            df_synth, synth_load, synth_row,
            SYNTH_PROMPT, SYNTH_EMOTIONS, SYNTH_ALIASES, output_path=OUTPUT_SYNTH, tag="Synth",
        )
else:
    df_synth_res = run_inference(
        df_synth, synth_load, synth_row,
        SYNTH_PROMPT, SYNTH_EMOTIONS, SYNTH_ALIASES, output_path=OUTPUT_SYNTH, tag="Synth",
    )

### Evaluation

In [ ]:
synth_metrics = evaluate_llava_results(
    df_synth_res, SYNTH_EMOTIONS,
    tag="SynthCAER test",
    save_stem="llava_synth",
)

### K-fold Robust analysis (full dataset)

In [ ]:
df_kf_synth = df_synth_res[~df_synth_res["unparseable"]].reset_index(drop=True)
print(f"Cases per fold: {len(df_kf_synth):,}")
print(df_kf_synth["true_emotion"].value_counts().to_string())

df_folds_synth = run_kfold(df_kf_synth, SYNTH_EMOTIONS, n_folds=N_FOLDS, random_seed=RANDOM_SEED)
print(df_folds_synth.to_string(index=False, float_format="{:.3f}".format))

In [ ]:
metric_cols = ["Accuracy", "Macro F1"] + [f"F1_{e}" for e in SYNTH_EMOTIONS]
df_stats_synth = df_folds_synth[metric_cols].agg(["mean", "std", "min", "max"])
df_stats_synth.index = ["Media", "Std", "Min", "Max"]
print(df_stats_synth.T.to_string(float_format="{:.3f}".format))

In [ ]:
plot_kfold_results(
    df_folds_synth, SYNTH_EMOTIONS,
    tag="SynthCAER",
    save_prefix="llava_synth",
)